# Learner Activity: Retail Performance Regression

Use the retail performance dataset to repeat the correlation and simple linear regression workflow from the lecture notebook.


In [100]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

DATA_FILE = "retail-performance-activity-data.csv"


## 1. Load And Inspect The Data

Load the CSV, preview the first few rows, and identify which columns are numeric.


In [101]:
df = pd.read_csv(DATA_FILE)
df.head()


,month,campaign_name,ad_spend,store_visits,transactions,avg_order_value,discount_rate,sales_revenue
0,2025-01,New Year Sale,18000,42000,3150,42.5,0.12,134800
1,2025-02,Loyalty Push,19500,43800,3295,43.1,0.10,141900
2,2025-03,Spring Refresh,21000,46100,3460,44.0,0.11,152600
3,2025-04,Weekend Deals,22500,47200,3515,44.7,0.13,156100
4,2025-05,Member Month,24100,49800,3720,45.2,0.09,168400


In [102]:
# Identify the numeric columns you want to compare.
# Use the answer key column order: ad_spend, store_visits, transactions,
# avg_order_value, discount_rate, and sales_revenue.
numeric_columns = [ "ad_spend", "store_visits", "transactions",
                   "avg_order_value", "discount_rate", "sales_revenue"]

# Use the numeric_columns list to select only those columns from df.
# Store the result in a DataFrame named numeric_df.
numeric_df = df[numeric_columns]
numeric_df.head()


,ad_spend,store_visits,transactions,avg_order_value,discount_rate,sales_revenue
0,18000,42000,3150,42.5,0.12,134800
1,19500,43800,3295,43.1,0.10,141900
2,21000,46100,3460,44.0,0.11,152600
3,22500,47200,3515,44.7,0.13,156100
4,24100,49800,3720,45.2,0.09,168400


## 2. Manual Pearson Correlation

Write a function that manually calculates Pearson correlation between two numeric arrays. Then use it to compare each numeric column against `sales_revenue`.


In [103]:
def pearson_correlation(x: np.ndarray, y: np.ndarray) -> float:
    # Convert x and y to numeric NumPy arrays before doing math.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Calculate the mean of x and the mean of y.
    x_mean = x.mean()
    y_mean = y.mean()

    # Use the centered values to calculate the Pearson numerator.
    # Formula: sum((x - x_mean) * (y - y_mean))
    numerator = sum((x - x_mean) * (y - y_mean))

    # Use the centered values to calculate the Pearson denominator.
    # Formula: sqrt(sum((x - x_mean)^2) * sum((y - y_mean)^2))
    denominator = np.sqrt(np.sum((x - x_mean)**2) * np.sum((y - y_mean)**2))

    # Return the Pearson correlation value.
    return numerator / denominator


In [104]:
# Identify the target column you want to compare every numeric column against.
target = "sales_revenue"
target_values = numeric_df[target].to_numpy(dtype=float)

# Use an empty dictionary to collect each column's correlation with sales_revenue.
target_correlations = {}

# Loop through each numeric column.
for column in numeric_df.columns:
    # Identify the x values from the current column.
    column_values = numeric_df[column].to_numpy(dtype=float)

    # Use the pearson_correlation function.
    # Pass the current column values as x and sales_revenue values as y.
    correlation = pearson_correlation(column_values, target_values)

    # Store the result in the dictionary using the column name as the key.
    target_correlations[column] = correlation

# Convert the dictionary to a Series and sort from strongest to weakest.
target_correlation_series = pd.Series(target_correlations).sort_values(ascending=False)
target_correlation_series


sales_revenue      1.000000
transactions       0.999819
store_visits       0.999672
ad_spend           0.994390
avg_order_value    0.986387
discount_rate      0.594815
dtype: float64

## 3. Manual Correlation Matrix

Build a correlation matrix manually by looping through every pair of numeric columns.


In [105]:
def build_correlation_matrix(numeric_df: pd.DataFrame) -> pd.DataFrame:
    # Create an empty square DataFrame.
    # Use the numeric column names as both the row index and column labels.
    cols = numeric_df.columns
    correlation_matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)

    # Loop through each row variable.
    for row in cols:
        # Loop through each column variable.
        for column in cols:
            # Identify the two arrays to compare.
            row_values = numeric_df[row].to_numpy(dtype=float)
            column_values = numeric_df[column].to_numpy(dtype=float)

            # Use pearson_correlation to calculate the relationship.
            correlation = pearson_correlation(row_values, column_values)

            # Store the result where the row variable and column variable meet.
            correlation_matrix.loc[row, column] = correlation

    return correlation_matrix

# Call the function and display the first few rows.
correlation_matrix = build_correlation_matrix(numeric_df)
correlation_matrix.head()


,ad_spend,store_visits,transactions,avg_order_value,discount_rate,sales_revenue
ad_spend,1.000000,0.995688,0.994455,0.997803,0.556802,0.994390
store_visits,0.995688,1.000000,0.999882,0.988408,0.596984,0.999672
transactions,0.994455,0.999882,1.000000,0.986441,0.597476,0.999819
avg_order_value,0.997803,0.988408,0.986441,1.000000,0.540518,0.986387
discount_rate,0.556802,0.596984,0.597476,0.540518,1.000000,0.594815


In [106]:
# Build a Plotly heatmap of your manual correlation matrix.
# Identify the z values, x labels, and y labels from correlation_matrix.
import plotly.graph_objects as go

fig = go.Figure(
    data=go.Heatmap(
        z=correlation_matrix.to_numpy(dtype=float),
        x=correlation_matrix.columns.tolist(),
        y=correlation_matrix.index.tolist(),
        zmin=-1,
        zmax=1,
        colorscale="RdYlGn",
        text=np.round(correlation_matrix.to_numpy(dtype=float), 3),
        texttemplate="%{text}",
    )
)
fig.update_layout(title="Retail Performance Correlation Matrix", template="plotly_white")
import plotly.io as pio
pio.renderers.default = "vscode" 
fig.show()


## 4. Simple Linear Regression

Choose one useful predictor for `sales_revenue`, then manually calculate a simple linear regression model.


In [107]:
# Identify the predictor column.
# Use ad_spend first so your output can be compared with the answer key.
predictor ="ad_spend"

# Identify the predictor values as x and sales revenue as y.
x = df[predictor].to_numpy(dtype=float)
y = df["sales_revenue"].to_numpy(dtype=float)

# Use pearson_correlation to check the relationship between x and y.
correlation = pearson_correlation(x, y)
correlation


np.float64(0.9943898580946783)

In [108]:
def simple_linear_regression(x: np.ndarray, y: np.ndarray) -> dict:
    # Convert x and y to numeric NumPy arrays.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Calculate the mean of x and the mean of y.
    x_mean = x.mean()
    y_mean = y.mean()

    # Calculate the slope.
    # Formula: sum((x - x_mean) * (y - y_mean)) / sum((x - x_mean)^2)
    slope = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)

    # Calculate the intercept.
    # Formula: y_mean - (slope * x_mean)
    intercept = y_mean - (slope * x_mean)

    # Generate predictions using the regression equation.
    predictions = intercept + (slope * x)

    # Calculate residuals, SSE, SST, R2, and RMSE.
    residuals = y - predictions
    sse = np.sum(residuals**2)
    sst = np.sum((y - y_mean)**2)
    r_squared = 1 - (sse / sst)
    rmse = np.sqrt(sse / len(y))        

    # Return all results in a dictionary so later cells can use them by name.
    return {
        "slope": slope,
        "intercept": intercept,
        "predictions": predictions,
        "sse": sse,
        "sst": sst,
        "r_squared": r_squared,
        "rmse": rmse,
    }


In [109]:
# Run the simple_linear_regression function.
# Pass the predictor array as x and sales revenue as y.
result = simple_linear_regression(x, y)

print("Simple Linear Regression Results")
print("--------------------------------")
print("Model Coefficients")
print("------------------")
print(f"Slope: {result['slope']}")
print(f"Intercept: {result['intercept']}")
print(f"Equation: sales_revenue = {result['intercept']:.2f} + ({result['slope']:.4f} * ad_spend)")
print()
print("Model Evaluation Metrics")
print("------------------------")
print(f"SSE: {result['sse']}")
print(f"SST: {result['sst']}")
print(f"R2 (fit metric): {result['r_squared']}")
print(f"RMSE (error metric): {result['rmse']}")
print()
print("Relationship Metric")
print("-------------------")
print(f"Pearson r (ad_spend vs sales_revenue): {correlation}")


Simple Linear Regression Results
--------------------------------
Model Coefficients
------------------
Slope: 7.089176310415248
Intercept: 681.2003630587715
Equation: sales_revenue = 681.20 + (7.0892 * ad_spend)

Model Evaluation Metrics
------------------------
SSE: 225552500.56727916
SST: 20158756666.666668
R2 (fit metric): 0.9888111898815546
RMSE (error metric): 4335.440198404302

Relationship Metric
-------------------
Pearson r (ad_spend vs sales_revenue): 0.9943898580946783


In [110]:
# Sort the data by the predictor so the regression line is drawn from left to right.
plot_df = df.sort_values(predictor).copy()

# Match the prediction order to the sorted predictor values.
plot_df["predicted_sales_revenue"] = result["predictions"][np.argsort(x)]

# Create a scatter plot of the actual data.
fig = go.Figure()  
fig.add_trace(go.Scatter(
    x=plot_df[predictor],
    y=plot_df["sales_revenue"],
    mode="markers",
    name="Actual data",
    text=plot_df["campaign_name"],
    hovertemplate="Campaign: %{text}<br>ad_spend: %{x}<br>sales_revenue: %{y}<extra></extra>",
))

# Add a regression line using the predicted sales revenue values.
fig.add_trace(go.Scatter(
    x=plot_df[predictor],
    y=plot_df["predicted_sales_revenue"],
    mode="lines",
    name="Regression line",
    hovertemplate="ad_spend: %{x}<br>predicted sales_revenue: %{y}<extra></extra>",
    )
)

# Label the chart and axes clearly.
fig.update_layout(
    title="Ad Spend vs Sales Revenue — Simple Linear Regression",
    xaxis_title="Ad Spend",
    yaxis_title="Sales Revenue",
    template="plotly_white",
)
fig.show()


In [111]:
# Record the predictions in the original DataFrame.
df["predicted_sales_revenue"] = result["predictions"]
# Save the DataFrame with predictions to results.csv.
df.to_csv("results.csv", index=False)


## 5. Interpretation

Write 2 to 4 sentences answering:

- Which variable did you use to predict sales revenue?
- How strong is the relationship?
- What does the slope mean in business terms?
- Is this model useful for decision-making? Why or why not?

-------------------------------------------------------------
**Ad spend was used to predict sales revenue. The relationship is strong and positive (Pearson r close to 1), meaning that as ad spend increases, sales revenue tends to rise proportionally since pearson correlation is a statistical measure that tells how strongly 2 variables are related and whether the relationship is positive or negative. The calculated pearson correlation is 0.94444 meaning the relationship is strong and linear. The slope tells us that each additional dollar of ad spend is associated with a specific increase in sales revenue. For example, a slope of ~3.5 means every $1 more in ads predicts roughly $3.50 more in revenue. This model is useful as a baseline for budgeting decisions, though with only 12 data points, predictions should be treated as directional guidance rather than precise forecasts.**


## Bonus Challenge: Lagged Ad Spend

Calculate the correlation between `ad_spend` and future `sales_revenue` using 0-month, 1-month, 2-month, and 3-month lags. Which lag has the strongest relationship?


In [112]:
# Display the current-month ad_spend values before creating a lag.
non_lagged = df['ad_spend']
non_lagged


0     18000
1     19500
2     21000
3     22500
4     24100
5     25800
6     27200
7     28900
8     30500
9     32200
10    34800
11    36500
Name: ad_spend, dtype: int64

In [113]:
# Use the shift function to move ad_spend down by 1 row.
# This lets you compare earlier ad_spend with later sales_revenue.
lagged = df['ad_spend'].shift(1)
lagged


0         NaN
1     18000.0
2     19500.0
3     21000.0
4     22500.0
5     24100.0
6     25800.0
7     27200.0
8     28900.0
9     30500.0
10    32200.0
11    34800.0
Name: ad_spend, dtype: float64

In [114]:
# Use an empty dictionary to store each lag's correlation result.
lag_results = {}

# Loop through 0-month, 1-month, 2-month, and 3-month lags.
for lag in range(4):
    # Use the shift function to create lagged ad_spend values.
    # Drop the first lag rows so ad_spend and sales_revenue are aligned.
    lagged_ad_spend = df['ad_spend'].shift(lag).iloc[lag:].to_numpy(dtype=float)

    # Identify the matching sales_revenue values for the same rows.
    aligned_sales_revenue = df['sales_revenue'].iloc[lag:].to_numpy(dtype=float)

    # Use pearson_correlation to compare lagged ad_spend and aligned sales_revenue.
    lag_r = pearson_correlation(lagged_ad_spend, aligned_sales_revenue)

    # Store the result using a label such as lag_0_months.
    lag_results[f"lag_{lag}_months"] = lag_r

# Convert the dictionary to a Series and sort from strongest to weakest.
lag_series = pd.Series(lag_results).sort_values(ascending=False)
lag_series


lag_0_months    0.994390
lag_1_months    0.991886
lag_3_months    0.991215
lag_2_months    0.989354
dtype: float64

Write 1 to 2 sentences answering: Which lag has the strongest relationship with `sales_revenue`? What does that suggest about current-month ad spend versus earlier ad spend?


----------------------------------------------------------
**lag_0_months has the strongest relationship with sales_revenue (r = 0.994). This suggests that current-month ad spend is the best predictor of current-month sales revenue which means that advertising has an immediate effect rather than a delayed one. Earlier ad spend (lags 1–3) is slightly less correlated, though all lags remain very strongly related as they are all close to 1.**

In [115]:
# Plot the lagged correlations as a bar chart.
fig = go.Figure(
    data=go.Bar(
        x=lag_series.index.tolist(),
        y=lag_series.values.tolist(),
        text=np.round(lag_series.values, 3),
        textposition="auto",
    )
)
fig.update_layout(title="Lagged Ad Spend Correlation with Sales Revenue", template="plotly_white")
fig.show()


# Bonus Info on Lagged Relationships

Lagged relationships can be useful for forecasting, but they are not necessarily causal. Think about whether both ad spend and sales revenue might be moving together because of seasonality, growth, or promotions.


# MORE PREDICTORS ANALYSIS

**Predictor: discount_rate**

In [ ]:
predictor ="discount_rate"

x = df[predictor].to_numpy(dtype=float)
y = df["sales_revenue"].to_numpy(dtype=float)

correlation = pearson_correlation(x, y)
correlation


np.float64(0.594814758019866)

In [117]:
def simple_linear_regression(x: np.ndarray, y: np.ndarray) -> dict:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_mean = x.mean()
    y_mean = y.mean()

    slope = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)

    intercept = y_mean - (slope * x_mean)

    predictions = intercept + (slope * x)

    residuals = y - predictions
    sse = np.sum(residuals**2)
    sst = np.sum((y - y_mean)**2)
    r_squared = 1 - (sse / sst)
    rmse = np.sqrt(sse / len(y))        

    return {
        "slope": slope,
        "intercept": intercept,
        "predictions": predictions,
        "sse": sse,
        "sst": sst,
        "r_squared": r_squared,
        "rmse": rmse,
    }


In [118]:
result = simple_linear_regression(x, y)

print("Simple Linear Regression Results")
print("--------------------------------")
print("Model Coefficients")
print("------------------")
print(f"Slope: {result['slope']}")
print(f"Intercept: {result['intercept']}")
print(f"Equation: sales_revenue = {result['intercept']:.2f} + ({result['slope']:.4f} * discount_rate)")
print()
print("Model Evaluation Metrics")
print("------------------------")
print(f"SSE: {result['sse']}")
print(f"SST: {result['sst']}")
print(f"R2 (fit metric): {result['r_squared']}")
print(f"RMSE (error metric): {result['rmse']}")
print()
print("Relationship Metric")
print("-------------------")
print(f"Pearson r (discount_rate vs sales_revenue): {correlation}")


Simple Linear Regression Results
--------------------------------
Model Coefficients
------------------
Slope: 938846.5499485066
Intercept: 72178.47579814625
Equation: sales_revenue = 72178.48 + (938846.5499 * discount_rate)

Model Evaluation Metrics
------------------------
SSE: 13026495901.13285
SST: 20158756666.666668
R2 (fit metric): 0.3538045963582319
RMSE (error metric): 32947.55415951849

Relationship Metric
-------------------
Pearson r (discount_rate vs sales_revenue): 0.594814758019866


In [119]:
plot_df = df.sort_values(predictor).copy()

plot_df["predicted_sales_revenue"] = result["predictions"][np.argsort(x)]

fig = go.Figure()  
fig.add_trace(go.Scatter(
    x=plot_df[predictor],
    y=plot_df["sales_revenue"],
    mode="markers",
    name="Actual data",
    text=plot_df["campaign_name"],
    hovertemplate="Campaign: %{text}<br>discount_rate: %{x}<br>sales_revenue: %{y}<extra></extra>",
))

# Add a regression line using the predicted sales revenue values.
fig.add_trace(go.Scatter(
    x=plot_df[predictor],
    y=plot_df["predicted_sales_revenue"],
    mode="lines",
    name="Regression line",
    hovertemplate="discount_rate: %{x}<br>predicted sales_revenue: %{y}<extra></extra>",
    )
)

# Label the chart and axes clearly.
fig.update_layout(
    title="Discount Rate vs Sales Revenue — Simple Linear Regression",
    xaxis_title="Discount Rate",
    yaxis_title="Sales Revenue",
    template="plotly_white",
)
fig.show()


In [120]:
df["predicted_sales_revenue"] = result["predictions"]
df.to_csv("results2.csv", index=False)

## 5. Interpretation

Write 2 to 4 sentences answering:

- Which variable did you use to predict sales revenue?
- How strong is the relationship?
- What does the slope mean in business terms?
- Is this model useful for decision-making? Why or why not?

-------------------------------------------------------------
**Discount rate was used to predict sales revenue. The relationship is weak (Pearson r = 0.595, R² = 0.354), meaning discount rate only explains about 35% of the variation in sales revenue. The slope of ~938,847 means that each 1-unit increase in discount rate predicts roughly $938,847 more in revenue however since discount rates only vary by small decimals in this dataset, the practical effect per percentage point is modest. This model is not very useful for decision-making on its own.**